In [57]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [58]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


In [59]:
train['HomePlanet'] = train['HomePlanet'].fillna(train['HomePlanet'].mode()[0])

train['CryoSleep'] = train['CryoSleep'].fillna(train['CryoSleep'].mode()[0])

train['Destination'] = train['Destination'].fillna(train['Destination'].mode()[0])

train['Age'] = train['Age'].fillna(train['Age'].median())

train['VIP'] = train['VIP'].fillna(train['VIP'].mode()[0])

In [60]:
train[['deck', 'num', 'side']] = train['Cabin'].str.split('/', expand=True)

In [61]:
train['deck'] = train['deck'].fillna(train['deck'].mode()[0])

train['num'] = pd.to_numeric(train['num'])
train['num'] = train['num'].fillna(train['num'].median())

train['side'] = train['side'].fillna(train['side'].mode()[0])

train = train.drop('Cabin', axis=1)

In [62]:
train[train['CryoSleep'] == True][['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].describe()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,2969.0,2967.0,2941.0,2972.0,2975.0
mean,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0


In [63]:
columnas_gasto = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# CryoSleep = True
for col in columnas_gasto:
    train.loc[(train['CryoSleep'] == True) & (train[col].isnull()), col] = 0

# CryoSleep = False
for col in columnas_gasto:
    mediana_despiertos = train[train['CryoSleep'] == False][col].median()
    train.loc[(train['CryoSleep'] == False) & (train[col].isnull()), col] = mediana_despiertos

In [65]:
train['totalSpend'] = train['RoomService'] + train['FoodCourt'] + train['ShoppingMall'] + train['Spa'] + train['VRDeck']

In [66]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8693 non-null   str    
 2   CryoSleep     8693 non-null   object 
 3   Destination   8693 non-null   str    
 4   Age           8693 non-null   float64
 5   VIP           8693 non-null   object 
 6   RoomService   8693 non-null   float64
 7   FoodCourt     8693 non-null   float64
 8   ShoppingMall  8693 non-null   float64
 9   Spa           8693 non-null   float64
 10  VRDeck        8693 non-null   float64
 11  Name          8493 non-null   str    
 12  Transported   8693 non-null   bool   
 13  deck          8693 non-null   str    
 14  num           8693 non-null   float64
 15  side          8693 non-null   str    
 16  totalSpend    8693 non-null   float64
dtypes: bool(1), float64(8), object(2), str(6)
memory usage: 1.4+ MB


In [82]:
# One hot encoding
train = pd.get_dummies(train, columns=['HomePlanet'], drop_first=True)

train = pd.get_dummies(train, columns=['Destination'], drop_first=True)

train = pd.get_dummies(train, columns=['deck'], drop_first=True)

# Convertir a boolean
mapeo = {'P': True, 'S': False}
train['side'] = train['side'].map(mapeo)

# Convertir a numerico
train['Age'] = pd.to_numeric(train['Age'], errors='coerce')

train['RoomService'] = pd.to_numeric(train['RoomService'], errors='coerce')

train['FoodCourt'] = pd.to_numeric(train['FoodCourt'], errors='coerce')

train['ShoppingMall'] = pd.to_numeric(train['ShoppingMall'], errors='coerce')

train['Spa'] = pd.to_numeric(train['Spa'], errors='coerce')

train['VRDeck'] = pd.to_numeric(train['VRDeck'], errors='coerce')

train['num'] = pd.to_numeric(train['num'], errors='coerce')

train['totalSpend'] = pd.to_numeric(train['totalSpend'], errors='coerce')

In [85]:
train = train.drop(['PassengerId', 'Name'], axis=1)

In [86]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  8693 non-null   object 
 1   Age                        8693 non-null   float64
 2   VIP                        8693 non-null   object 
 3   RoomService                8693 non-null   float64
 4   FoodCourt                  8693 non-null   float64
 5   ShoppingMall               8693 non-null   float64
 6   Spa                        8693 non-null   float64
 7   VRDeck                     8693 non-null   float64
 8   Transported                8693 non-null   bool   
 9   num                        8693 non-null   float64
 10  side                       8693 non-null   bool   
 11  totalSpend                 8693 non-null   float64
 12  HomePlanet_Europa          8693 non-null   bool   
 13  HomePlanet_Mars            8693 non-null   bool   
 14  Des

In [87]:
# Division de variables
X = train.drop('Transported', axis=1)
y = train['Transported']

In [88]:
# Preparación de entrenamiento y validación
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [89]:
# Entrenamiento del modelo
from sklearn.ensemble import RandomForestClassifier

modelo = RandomForestClassifier(random_state=42)
modelo.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [90]:
# Primer resultado de validación
from sklearn.metrics import accuracy_score

predicciones = modelo.predict(X_val)
precision = accuracy_score(y_val, predicciones)
print(f"Precisión del modelo: {precision:.2%}")

Precisión del modelo: 79.47%


In [91]:
y_train.value_counts(normalize=True)
y_val.value_counts(normalize=True)

Transported
True     0.504888
False    0.495112
Name: proportion, dtype: float64

In [92]:
importancias = pd.Series(modelo.feature_importances_, index=X_train.columns)
importancias = importancias.sort_values(ascending=False)
print(importancias)

num                          0.169301
totalSpend                   0.139170
Age                          0.121247
VRDeck                       0.084159
Spa                          0.082423
FoodCourt                    0.078528
RoomService                  0.074821
ShoppingMall                 0.064014
CryoSleep                    0.054122
side                         0.020892
deck_G                       0.018244
HomePlanet_Europa            0.015941
Destination_TRAPPIST-1e      0.013282
deck_F                       0.013042
HomePlanet_Mars              0.012488
deck_E                       0.011992
deck_C                       0.007944
deck_B                       0.006578
Destination_PSO J318.5-22    0.006333
deck_D                       0.003304
VIP                          0.002089
deck_T                       0.000086
dtype: float64


In [94]:
from sklearn.model_selection import cross_val_score

modelo_cv = RandomForestClassifier(random_state=42)
resultados_cv = cross_val_score(modelo_cv, X, y, cv=5)

print(f"Precisión en cada partición: {resultados_cv}")
print(f"Precisión promedio: {resultados_cv.mean():.2%}")
print(f"Desviación estándar: {resultados_cv.std():.2%}")

Precisión en cada partición: [0.76193214 0.7573318  0.80506038 0.81127733 0.78883774]
Precisión promedio: 78.49%
Desviación estándar: 2.19%


In [95]:
from sklearn.model_selection import GridSearchCV

parametros = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

busqueda = GridSearchCV(RandomForestClassifier(random_state=42), parametros, cv=5)
busqueda.fit(X, y)

print(f"Mejores parámetros: {busqueda.best_params_}")
print(f"Mejor precisión promedio: {busqueda.best_score_:.2%}")

Mejores parámetros: {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 100}
Mejor precisión promedio: 79.77%


In [96]:
# Cargar el dataframe de entrenamiento y test
test_original = pd.read_csv("data/test.csv")
test_ids = test_original['PassengerId']

train_original = pd.read_csv("data/train.csv")

test_clean = test_original.copy()

# Rellenar valores faltantes con la moda o mediana de train_original
test_clean['HomePlanet'] = test_clean['HomePlanet'].fillna(train_original['HomePlanet'].mode()[0])

test_clean['CryoSleep'] = test_clean['CryoSleep'].fillna(train_original['CryoSleep'].mode()[0])

test_clean['Destination'] = test_clean['Destination'].fillna(train_original['Destination'].mode()[0])

test_clean['Age'] = test_clean['Age'].fillna(train_original['Age'].median())

test_clean['VIP'] = test_clean['VIP'].fillna(train_original['VIP'].mode()[0]) 


# Dividimos la columna Cabin en test_clean y train_original para obtener la moda de deck
test_clean[['deck', 'num', 'side']] = test_clean['Cabin'].str.split('/', expand=True)

train_original[['deck', 'num', 'side']] = train_original['Cabin'].str.split('/', expand=True)


# Rellenar valores faltantes con la moda o mediana de train_original y eliminamos Cabin
test_clean['deck'] = test_clean['deck'].fillna(train_original['deck'].mode()[0])

# Convertimos a numerico y rellenamos valores faltantes con la mediana de train_original
test_clean['num'] = pd.to_numeric(test_clean['num'])
train_original['num'] = pd.to_numeric(train_original['num'])

test_clean['num'] = test_clean['num'].fillna(train_original['num'].median())

test_clean['side'] = test_clean['side'].fillna(train_original['side'].mode()[0])

test_clean = test_clean.drop('Cabin', axis=1)


# Crear la columna totalSpend en test_clean
columnas_gasto = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# CryoSleep = True
for col in columnas_gasto:
    test_clean.loc[(test_clean['CryoSleep'] == True) & (test_clean[col].isnull()), col] = 0

# CryoSleep = False
for col in columnas_gasto:
    # Usamos la mediana de train_original para rellenar los valores faltantes en test_clean
    mediana_despiertos = train_original[train_original['CryoSleep'] == False][col].median()
    test_clean.loc[(test_clean['CryoSleep'] == False) & (test_clean[col].isnull()), col] = mediana_despiertos


# Nueva columna para el total de gastos de servicios
test_clean['totalSpend'] = test_clean['RoomService'] + test_clean['FoodCourt'] + test_clean['ShoppingMall'] + test_clean['Spa'] + test_clean['VRDeck']


# One hot encoding
test_clean = pd.get_dummies(test_clean, columns=['HomePlanet'], drop_first=True)

test_clean = pd.get_dummies(test_clean, columns=['Destination'], drop_first=True)

test_clean = pd.get_dummies(test_clean, columns=['deck'], drop_first=True)


# Convertir a boolean
mapeo = {'P': True, 'S': False}
test_clean['side'] = test_clean['side'].map(mapeo)


# Convertir a numerico
test_clean['Age'] = pd.to_numeric(test_clean['Age'], errors='coerce')

test_clean['RoomService'] = pd.to_numeric(test_clean['RoomService'], errors='coerce')

test_clean['FoodCourt'] = pd.to_numeric(test_clean['FoodCourt'], errors='coerce')

test_clean['ShoppingMall'] = pd.to_numeric(test_clean['ShoppingMall'], errors='coerce')

test_clean['Spa'] = pd.to_numeric(test_clean['Spa'], errors='coerce')

test_clean['VRDeck'] = pd.to_numeric(test_clean['VRDeck'], errors='coerce')

test_clean['num'] = pd.to_numeric(test_clean['num'], errors='coerce')

test_clean['totalSpend'] = pd.to_numeric(test_clean['totalSpend'], errors='coerce')


# Eliminamos columnas que no se usarán para la predicción
test_model = test_clean.drop(['PassengerId', 'Name'], axis=1)

# Aseguramos que las columnas de test_model coincidan con las de X
test_model = test_model[X.columns]


In [97]:
# Entrenamos el modelo final con los mejores parámetros encontrados y hacemos predicciones sobre el conjunto de test
modelo_final = RandomForestClassifier(random_state=42, **busqueda.best_params_)
modelo_final.fit(X, y)

predicciones_test = modelo_final.predict(test_model)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': predicciones_test
})

submission.to_csv('submission.csv', index=False)